In [1]:
import os
import json
import math
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import scipy.io
import h5py

In [2]:
ROOT = Path("./lab5_svhn")
RAW_DIR = ROOT / "raw"
PROC_DIR = ROOT / "processed"

TRAIN_DIR = RAW_DIR / "train"
TEST_DIR = RAW_DIR / "test"

YOLO_DIR = PROC_DIR / "svhn_number_yolo"

IMG_SIZE = 640
SEED = 42

random.seed(SEED)
np.random.seed(SEED)

YOLO_MODEL = "yolov8n.pt"
EPOCHS = 40
BATCH = 16

In [3]:
for p in [
    YOLO_DIR / "images" / "train",
    YOLO_DIR / "images" / "val",
    YOLO_DIR / "images" / "test",
    YOLO_DIR / "labels" / "train",
    YOLO_DIR / "labels" / "val",
    YOLO_DIR / "labels" / "test",
]:
    p.mkdir(parents=True, exist_ok=True)

print("Folders created.")

Folders created.


In [4]:
#check dir

print("TRAIN_DIR exists:", TRAIN_DIR.exists())
print("TEST_DIR exists:", TEST_DIR.exists())

print("digitStruct train exists:", (TRAIN_DIR / "digitStruct.mat").exists())
print("digitStruct test exists:", (TEST_DIR / "digitStruct.mat").exists())

print("Num train images:", len(list(TRAIN_DIR.glob("*.png"))))
print("Num test images:", len(list(TEST_DIR.glob("*.png"))))

TRAIN_DIR exists: True
TEST_DIR exists: True
digitStruct train exists: True
digitStruct test exists: True
Num train images: 33402
Num test images: 13068


In [5]:
def _resolve_h5_value(v, mat_file):
    if isinstance(v, h5py.Reference):
        data = mat_file[v][()]
    else:
        data = v

    data = np.array(data)

    # ép về scalar
    if data.size == 1:
        return float(data.squeeze())

    # fallback nếu shape lạ
    return float(data.reshape(-1)[0])


def _read_attr(box_group, mat_file, key):
    value = box_group[key]
    result = []

    # value thường có shape (n,1)
    for i in range(value.shape[0]):
        v = value[i][0]
        result.append(_resolve_h5_value(v, mat_file))

    return result


def get_box_data(mat_file, box_ref):
    box_group = mat_file[box_ref]
    attrs = {}

    for key in ["label", "left", "top", "width", "height"]:
        attrs[key] = _read_attr(box_group, mat_file, key)

    return attrs


def load_svhn_annotations(split_dir):
    mat_path = Path(split_dir) / "digitStruct.mat"
    data = []

    with h5py.File(mat_path, "r") as f:
        names = f["digitStruct"]["name"]
        bboxes = f["digitStruct"]["bbox"]

        for i in range(len(names)):
            name_ref = names[i][0]
            filename = "".join(chr(c[0]) for c in f[name_ref][()])
            bbox_ref = bboxes[i][0]

            box_data = get_box_data(f, bbox_ref)

            data.append({
                "filename": filename,
                "labels": [int(x) for x in box_data["label"]],
                "left": [float(x) for x in box_data["left"]],
                "top": [float(x) for x in box_data["top"]],
                "width": [float(x) for x in box_data["width"]],
                "height": [float(x) for x in box_data["height"]],
            })

    return data

In [6]:
train_ann = load_svhn_annotations(TRAIN_DIR)
test_ann = load_svhn_annotations(TEST_DIR)

print("Train images:", len(train_ann))
print("Test images:", len(test_ann))
print("Sample annotation:", train_ann[0])

Train images: 33402
Test images: 13068
Sample annotation: {'filename': '1.png', 'labels': [1, 9], 'left': [246.0, 323.0], 'top': [77.0, 81.0], 'width': [81.0, 96.0], 'height': [219.0, 219.0]}


In [7]:
def merge_digit_boxes(record):
    lefts = np.array(record["left"], dtype=float)
    tops = np.array(record["top"], dtype=float)
    widths = np.array(record["width"], dtype=float)
    heights = np.array(record["height"], dtype=float)

    x_min = lefts.min()
    y_min = tops.min()
    x_max = (lefts + widths).max()
    y_max = (tops + heights).max()

    return {
        "filename": record["filename"],
        "x_min": float(x_min),
        "y_min": float(y_min),
        "x_max": float(x_max),
        "y_max": float(y_max),
        "num_digits": len(record["labels"]),
        "labels": record["labels"]
    }

train_merged = [merge_digit_boxes(r) for r in train_ann]
test_merged = [merge_digit_boxes(r) for r in test_ann]

print("Sample merged:", train_merged[0])

Sample merged: {'filename': '1.png', 'x_min': 246.0, 'y_min': 77.0, 'x_max': 419.0, 'y_max': 300.0, 'num_digits': 2, 'labels': [1, 9]}


In [8]:
def show_sample_bbox(split_dir, merged_records, idx=0):
    rec = merged_records[idx]
    img_path = Path(split_dir) / rec["filename"]

    img = Image.open(img_path).convert("RGB")
    img_np = np.array(img)

    plt.figure(figsize=(6, 6))
    plt.imshow(img_np)
    ax = plt.gca()

    rect = plt.Rectangle(
        (rec["x_min"], rec["y_min"]),
        rec["x_max"] - rec["x_min"],
        rec["y_max"] - rec["y_min"],
        fill=False,
        edgecolor="red",
        linewidth=2
    )
    ax.add_patch(rect)

    plt.title(f"{rec['filename']} | digits={rec['num_digits']} | labels={rec['labels']}")
    plt.axis("off")
    plt.show()

In [9]:
indices = list(range(len(train_merged)))
random.shuffle(indices)

n_total = len(indices)
n_train = int(0.9 * n_total)
n_val = n_total - n_train

train_idx = indices[:n_train]
val_idx = indices[n_train:]

train_final = [train_merged[i] for i in train_idx]
val_final = [train_merged[i] for i in val_idx]
test_final = test_merged

print("Train:", len(train_final))
print("Val:", len(val_final))
print("Test:", len(test_final))

Train: 30061
Val: 3341
Test: 13068


In [10]:
def to_yolo_bbox(img_w, img_h, x_min, y_min, x_max, y_max):
    x_center = ((x_min + x_max) / 2.0) / img_w
    y_center = ((y_min + y_max) / 2.0) / img_h
    width = (x_max - x_min) / img_w
    height = (y_max - y_min) / img_h
    return x_center, y_center, width, height

In [11]:
def export_split(records, src_dir, split_name):
    img_out_dir = YOLO_DIR / "images" / split_name
    lbl_out_dir = YOLO_DIR / "labels" / split_name

    for rec in records:
        src_img = Path(src_dir) / rec["filename"]
        dst_img = img_out_dir / rec["filename"]
        shutil.copy(src_img, dst_img)

        img = Image.open(src_img)
        w, h = img.size

        x_c, y_c, bw, bh = to_yolo_bbox(
            w, h,
            rec["x_min"], rec["y_min"],
            rec["x_max"], rec["y_max"]
        )

        label_path = lbl_out_dir / (Path(rec["filename"]).stem + ".txt")
        with open(label_path, "w", encoding="utf-8") as f:
            f.write(f"0 {x_c:.6f} {y_c:.6f} {bw:.6f} {bh:.6f}\n")


export_split(train_final, TRAIN_DIR, "train")
export_split(val_final, TRAIN_DIR, "val")
export_split(test_final, TEST_DIR, "test")

print("YOLO export done.")

YOLO export done.


In [12]:
yaml_text = f"""
path: {YOLO_DIR.as_posix()}
train: images/train
val: images/val
test: images/test

names:
  0: number
""".strip()

yaml_path = YOLO_DIR / "svhn_number.yaml"
yaml_path.write_text(yaml_text, encoding="utf-8")

print(yaml_path.read_text())

path: lab5_svhn/processed/svhn_number_yolo
train: images/train
val: images/val
test: images/test

names:
  0: number


In [13]:
sample_label_files = list((YOLO_DIR / "labels" / "train").glob("*.txt"))[:5]

for lf in sample_label_files:
    print(lf.name, "->", lf.read_text().strip())

1.txt -> 0 0.448718 0.538571 0.233468 0.637143
10.txt -> 0 0.493243 0.500000 0.310811 0.783784
100.txt -> 0 0.447761 0.425926 0.358209 0.851852
1000.txt -> 0 0.500000 0.476190 0.227273 0.857143
10000.txt -> 0 0.485401 0.580645 0.313869 0.516129


In [14]:
print("Train images:", len(list((YOLO_DIR / "images" / "train").glob("*.png"))))
print("Val images:", len(list((YOLO_DIR / "images" / "val").glob("*.png"))))
print("Test images:", len(list((YOLO_DIR / "images" / "test").glob("*.png"))))

print("Train labels:", len(list((YOLO_DIR / "labels" / "train").glob("*.txt"))))
print("Val labels:", len(list((YOLO_DIR / "labels" / "val").glob("*.txt"))))
print("Test labels:", len(list((YOLO_DIR / "labels" / "test").glob("*.txt"))))

Train images: 30061
Val images: 3341
Test images: 13068
Train labels: 30061
Val labels: 3341
Test labels: 13068


In [15]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(yaml_path),
    epochs=10,
    imgsz=416,
    batch=4,
    project="lab5_runs",
    name="svhn_number_detector",
    device="cpu"
)

Ultralytics 8.4.37  Python-3.14.0 torch-2.10.0+cpu CPU (AMD Ryzen 7 5700X 8-Core Processor)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=lab5_svhn\processed\svhn_number_yolo\svhn_number.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=416, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=svhn_number_detector2, nbs=64, nms=False, opset=None, optimize=False, o

KeyboardInterrupt: 